In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Instrument calibration: BEER at ESS

This example demonstrates a Rietveld refinement of a duplex steel
structure using time-of-flight neutron powder diffraction data
simulated with McStas.

Two datasets from two symmetrically positioned banks (S2 and N2) of
the BEER instrument are analyzed in this tutorial.

## Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data
from easydiffraction import extract_data_paths_from_zip
from easydiffraction import extract_metadata

## Define Structures

This section covers how to add structures and modify their
parameters.

#### Create Ferrite Structure

In [3]:
ferrite = StructureFactory.from_scratch(name='ferrite')

ferrite.space_group.name_h_m = 'I m -3 m'
ferrite.space_group.it_coordinate_system_code = '1'

ferrite.cell.length_a = 2.886

ferrite.atom_sites.create(
    label='Fe',
    type_symbol='Fe',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=1.0,
)

#### Create Austenite Structure

In [4]:
austenite = StructureFactory.from_scratch(name='austenite')

austenite.space_group.name_h_m = 'F m -3 m'
austenite.space_group.it_coordinate_system_code = '1'

austenite.cell.length_a = 3.6468

austenite.atom_sites.create(
    label='Fe',
    type_symbol='Fe',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=1.0,
)

## Define Experiments

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

#### Download Measured Data

In [5]:
zip_path = download_data(id=33, destination='data')
data_paths = extract_data_paths_from_zip(zip_path, destination='data/ed-20')

data_path_s2 = data_paths[1]  # 'Duplex_in_HR_for_IRF_S2.dat'
data_path_n2 = data_paths[0]  # 'Duplex_in_HR_for_IRF_N2.dat'

Getting data...


Data #33: ferrite + austenite, BEER (ESS), S2 and N2 detector bank datasets


✅ Data #33 downloaded to 'data/ed-33.zip'


#### Create Experiment

In [6]:
expt_s2 = ExperimentFactory.from_data_path(
    name='expt_s2',
    data_path=data_path_s2,
    beam_mode='time-of-flight',
)

In [7]:
expt_n2 = ExperimentFactory.from_data_path(
    name='expt_n2',
    data_path=data_path_n2,
    beam_mode='time-of-flight',
)

#### Set Instrument

In [8]:
expt_s2.instrument.setup_twotheta_bank = extract_metadata(
    data_path_s2, r'two_theta\s*=\s*(\d*\.?\d+)'
)
expt_s2.instrument.calib_d_to_tof_linear = extract_metadata(
    data_path_s2, r'DIFC\s*=\s*(\d*\.?\d+)'
)

In [9]:
expt_n2.instrument.setup_twotheta_bank = extract_metadata(
    data_path_n2, r'two_theta\s*=\s*(\d*\.?\d+)'
)
expt_n2.instrument.calib_d_to_tof_linear = extract_metadata(
    data_path_n2, r'DIFC\s*=\s*(\d*\.?\d+)'
)

#### Set Peak Profile

In [10]:
expt_s2.show_peak_profile_types()

Peak profile types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


In [11]:
expt_s2.peak_profile_type = 'pseudo-voigt'

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'expt_s2' changed to


pseudo-voigt


In [12]:
expt_s2.peak.broad_gauss_sigma_0 = 300
expt_s2.peak.broad_gauss_sigma_1 = 1200
expt_s2.peak.broad_gauss_sigma_2 = 900

In [13]:
expt_n2.peak_profile_type = 'pseudo-voigt'

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'expt_n2' changed to


pseudo-voigt


In [14]:
expt_n2.peak.broad_gauss_sigma_0 = 300
expt_n2.peak.broad_gauss_sigma_1 = 1200
expt_n2.peak.broad_gauss_sigma_2 = 900

#### Set Background

In [15]:
expt_s2.show_background_types()

Background types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,chebyshev,Chebyshev polynomial background
2,*,line-segment,Linear interpolation between points


In [16]:
# expt_s2.background_type = 'line-segment'

In [17]:
for idx, (x, y) in enumerate(
    [
        (40111.8789, 0.0170),
        (41193.5664, 0.1484),
        (42041.3750, 0.1848),
        (42713.7734, 0.1975),
        (44409.3945, 0.1891),
        (45198.7344, 0.2147),
        (46251.1875, 0.1887),
        (49350.0742, 0.2194),
        (51289.6836, 0.1991),
        (55245.1992, 0.1981),
        (55679.7070, 0.2276),
        (56383.9102, 0.2439),
        (58956.1797, 0.2907),
        (61536.4570, 0.3067),
        (63768.0469, 0.3242),
        (65581.2109, 0.2973),
        (70183.8516, 0.2575),
        (71787.8203, 0.2321),
        (78343.1094, 0.2158),
        (80016.8047, 0.1694),
        (98141.8516, 0.2400),
        (99262.2344, 0.4335),
        (100985.8516, 0.4375),
        (101933.8516, 0.3427),
        (108656.0312, 0.5339),
        (110896.7500, 0.9537),
        (113137.4844, 1.1668),
        (114430.2031, 1.1164),
        (116929.4844, 0.9161),
        (119428.7422, 0.6885),
        (134506.3438, 0.0692),
    ],
    start=1,
):
    expt_s2.background.create(id=str(idx), x=x, y=y)

In [18]:
for point in expt_s2.background:
    expt_n2.background.create(id=point.id.value, x=point.x.value, y=point.y.value)

#### Set Linked Phases

In [19]:
expt_s2.linked_phases.create(id='ferrite', scale=10)
expt_s2.linked_phases.create(id='austenite', scale=10)

In [20]:
expt_n2.linked_phases.create(id='ferrite', scale=10)
expt_n2.linked_phases.create(id='austenite', scale=10)

#### Set Excluded Regions

In [21]:
expt_s2.excluded_regions.create(id='1', start=0, end=40500)
expt_s2.excluded_regions.create(id='2', start=130000, end=180000)

In [22]:
expt_n2.excluded_regions.create(id='1', start=0, end=40500)
expt_n2.excluded_regions.create(id='2', start=130000, end=180000)

## Define Project

The project object is used to manage the structure, experiments,
and analysis

#### Create Project

In [23]:
project = Project(name='beer')
project.save_as(dir_path='beer_mcstas')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saving project 📦 'beer' to


/home/runner/work/diffraction-lib/diffraction-lib/docs/docs/tutorials/beer_mcstas


├── 📄 project.cif


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📄 summary.cif


#### Add Structures

In [24]:
project.structures.add(ferrite)
project.structures.add(austenite)

#### Add Experiments

In [25]:
project.experiments.add(expt_s2)
project.experiments.add(expt_n2)

#### Plot Measured vs Calculated

In [26]:
project.display.plotter.plot_meas_vs_calc(expt_name='expt_s2', show_residual=False)

In [27]:
project.display.plotter.plot_meas_vs_calc(expt_name='expt_n2', show_residual=False)

## Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

#### Set Fit Mode

In [28]:
project.analysis.fit.show_modes()

Fit modes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,*,single,Independent fitting of each experiment
2,,joint,Simultaneous fitting of all experiments with weights
3,,sequential,Sequential fitting over data files in a directory


In [29]:
project.analysis.fit.mode = 'joint'

#### Set Free Parameters

In [30]:
project.analysis.display.fittable_params()

Fittable parameters for all structures (🧩 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,units,free
1,ferrite,cell,,length_a,2.88600,,Å,False
2,ferrite,cell,,length_b,2.88600,,Å,False
3,ferrite,cell,,length_c,2.88600,,Å,False
4,ferrite,cell,,angle_alpha,90.00000,,deg,False
5,ferrite,cell,,angle_beta,90.00000,,deg,False
6,ferrite,cell,,angle_gamma,90.00000,,deg,False
7,ferrite,atom_site,Fe,fract_x,0.00000,,,False
8,ferrite,atom_site,Fe,fract_y,0.00000,,,False
9,ferrite,atom_site,Fe,fract_z,0.00000,,,False
10,ferrite,atom_site,Fe,occupancy,1.00000,,,False


Fittable parameters for all experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,units,free
1,expt_s2,linked_phases,ferrite,scale,10.00000,,,False
2,expt_s2,linked_phases,austenite,scale,10.00000,,,False
3,expt_s2,peak,,lorentz_gamma_0,0.00000,,μs,False
4,expt_s2,peak,,lorentz_gamma_1,0.00000,,μs/Å,False
5,expt_s2,peak,,lorentz_gamma_2,0.00000,,μs²/Å²,False
6,expt_s2,peak,,gauss_sigma_0,300.00000,,μs²,False
7,expt_s2,peak,,gauss_sigma_1,1200.00000,,μs/Å,False
8,expt_s2,peak,,gauss_sigma_2,900.00000,,μs²/Å²,False
9,expt_s2,instrument,,twotheta_bank,90.00000,,deg,False
10,expt_s2,instrument,,d_to_tof_offset,0.00000,,μs,False


In [31]:
ferrite.atom_sites['Fe'].adp_iso.free = True
austenite.atom_sites['Fe'].adp_iso.free = True

In [32]:
expt_s2.linked_phases['ferrite'].scale.free = True
expt_s2.linked_phases['austenite'].scale.free = True

expt_s2.peak.broad_gauss_sigma_0.free = True
expt_s2.peak.broad_gauss_sigma_1.free = True
expt_s2.peak.broad_gauss_sigma_2.free = True
expt_s2.peak.broad_lorentz_gamma_0.free = True

expt_s2.instrument.calib_d_to_tof_offset.free = True

for segment in expt_s2.background:
    segment.y.free = True

In [33]:
expt_n2.linked_phases['ferrite'].scale.free = True
expt_n2.linked_phases['austenite'].scale.free = True

expt_n2.peak.broad_gauss_sigma_0.free = True
expt_n2.peak.broad_gauss_sigma_1.free = True
expt_n2.peak.broad_gauss_sigma_2.free = True
expt_n2.peak.broad_lorentz_gamma_0.free = True

expt_n2.instrument.calib_d_to_tof_offset.free = True

for segment in expt_n2.background:
    segment.y.free = True

#### Add Constraints

In [34]:
project.analysis.aliases.create(
    label='s2_ferrite_scale', param=expt_s2.linked_phases['ferrite'].scale
)
project.analysis.aliases.create(
    label='s2_austenite_scale', param=expt_s2.linked_phases['austenite'].scale
)

project.analysis.aliases.create(
    label='n2_ferrite_scale', param=expt_n2.linked_phases['ferrite'].scale
)
project.analysis.aliases.create(
    label='n2_austenite_scale', param=expt_n2.linked_phases['austenite'].scale
)

project.analysis.constraints.create(expression='n2_ferrite_scale = s2_ferrite_scale')
project.analysis.constraints.create(expression='n2_austenite_scale = s2_austenite_scale')

#### Run Fitting

Run full fitting with all free parameters.

In [35]:
project.analysis.fit()

Using all experiments 🔬 ['expt_s2', 'expt_n2'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,843.64,
2,80,89.69,89.4% ↓
3,157,15.07,83.2% ↓
4,234,13.82,8.3% ↓
5,543,13.81,


<IPython.core.display.Javascript object>

...

│   └── 📄 expt_s2.cif


│   └── 📄 expt_n2.cif


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📄 summary.cif


Fix background and run fitting again.

In [36]:
for segment in expt_s2.background:
    segment.y.free = False
for segment in expt_n2.background:
    segment.y.free = False

In [37]:
project.analysis.fit()

Using all experiments 🔬 ['expt_s2', 'expt_n2'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,13.66,
2,19,13.66,


<IPython.core.display.Javascript object>

...

│   └── 📄 expt_s2.cif


│   └── 📄 expt_n2.cif


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📄 summary.cif


Show fit results and parameter correlations.

In [38]:
project.analysis.display.fit_results()
project.display.plotter.plot_param_correlations()

Fit results


✅ Success: True


⏱️ Fitting time: 1.26 seconds


📏 Goodness-of-fit (reduced χ²): 13.66


📏 R-factor (Rf): 5.05%


...

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,ferrite,atom_site,Fe,adp_iso,1.7213,1.7213,0.0131,Å²,0.00 % ↑
2,austenite,atom_site,Fe,adp_iso,1.7092,1.7092,0.0122,Å²,0.00 % ↑
3,expt_s2,linked_phases,ferrite,scale,50.3587,50.3587,0.1529,,0.00 % ↑
4,expt_s2,linked_phases,austenite,scale,11.9852,11.9852,0.0308,,0.00 % ↑
5,expt_s2,peak,,lorentz_gamma_0,5.0196,5.0196,0.1521,μs,0.00 % ↑
6,expt_s2,peak,,gauss_sigma_0,881.0993,881.1030,58.2232,μs²,0.00 % ↑
7,expt_s2,peak,,gauss_sigma_1,1298.8025,1298.7991,58.1755,μs/Å,0.00 % ↓
8,expt_s2,peak,,gauss_sigma_2,308.1175,308.1181,11.5800,μs²/Å²,0.00 % ↑
9,expt_s2,instrument,,d_to_tof_offset,-10.4444,-10.4444,0.1217,μs,0.00 % ↑
10,expt_n2,peak,,lorentz_gamma_0,5.0134,5.0134,0.1527,μs,0.00 % ↑


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Plot Measured vs Calculated

Show full range in TOF.

In [39]:
project.display.plotter.plot_meas_vs_calc(expt_name='expt_s2', show_residual=False)

In [40]:
project.display.plotter.plot_meas_vs_calc(expt_name='expt_n2', show_residual=False)

Show selected peaks in d-spacing.

In [41]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='expt_s2',
    x='d_spacing',
    x_min=2.08,
    x_max=2.13,
)

In [42]:
project.display.plotter.plot_meas_vs_calc(
    expt_name='expt_n2',
    x='d_spacing',
    x_min=2.08,
    x_max=2.13,
)